In [1]:
import numpy as np
import pandas as pd
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

import seaborn as sns
import matplotlib.pyplot as plt
import os
from datetime import date
import datetime

# Generating the training data for the Heat and Diffusion Model

In [2]:
data_dir = "./1D-AEMpy/"
depth_steps = 9 * 2 

print(os.getcwd())

D:\projects\1D-AEMpy\mcl\1_trainingData-MLP


In [3]:
meterological_data_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_meteorology_input.csv")
meterological_data_df = meterological_data_df # considering everything from 2nd time step

num_time_steps = meterological_data_df.shape[0]
depth_list = np.array(list(range(0, depth_steps)) * num_time_steps)*0.5+.25
depth_df = pd.DataFrame(data={'depth':depth_list})

#repeating the dataframe depth_steps number of times
meterological_data_df = pd.DataFrame(np.repeat(meterological_data_df.values, depth_steps, axis=0), columns=meterological_data_df.columns)
meterological_data_df = pd.concat([depth_df, meterological_data_df], ignore_index=False, axis=1)
meterological_data_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,icemovAvg,density_snow,ice_prior,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,21.360697,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,21.360697,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,21.360697,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,21.360697,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697


In [4]:
# Input INITIAL TEMP 00

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_temp_initial00.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_init00':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

,time,temp_init00,depth
0,2018-07-05 15:00:00,29.567320,0.25
1,2018-07-05 15:00:00,28.898525,0.75
2,2018-07-05 15:00:00,28.229730,1.25
3,2018-07-05 15:00:00,26.238430,1.75
4,2018-07-05 15:00:00,24.247130,2.25
...,...,...,...
788395,2023-07-04 14:00:00,6.408465,6.75
788396,2023-07-04 14:00:00,6.054016,7.25
788397,2023-07-04 14:00:00,5.857477,7.75
788398,2023-07-04 14:00:00,5.797751,8.25


In [5]:
final_df = meterological_data_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,density_snow,ice_prior,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,29.567320
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,28.898525
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,28.229730
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,26.238430
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,24.247130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,6.408465
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,6.054016
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,5.857477
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,5.797751


In [6]:
# Input HEAT TEMP 01

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_temp_heat01.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_heat01':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,ice_prior,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,29.567320,29.351169
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,28.898525,28.984686
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,28.229730,28.276431
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,26.238430,26.263232
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,24.247130,24.260344
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,6.408465,6.408659
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,6.054016,6.054170
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,5.857477,5.857625
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,5.797751,5.797938


In [7]:
# Input ICE TEMP 02

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_temp_ice02.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_ice02':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,29.567320,29.351169,29.351169
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,28.898525,28.984686,28.984686
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,28.229730,28.276431,28.276431
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,26.238430,26.263232,26.263232
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.0,250.0,1.0,0.0,0.8,7.216207,24.247130,24.260344,24.260344
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,6.408465,6.408659,6.408659
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,6.054016,6.054170,6.054170
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,5.857477,5.857625,5.857625
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.0,250.0,1.0,0.0,0.8,21.360697,5.797751,5.797938,5.797938


In [8]:
# Input DIFF TEMP 03

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_temp_diff03.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_diff03':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,250.0,1.0,0.0,0.8,7.216207,29.567320,29.351169,29.351169,29.351169
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,250.0,1.0,0.0,0.8,7.216207,28.898525,28.984686,28.984686,28.984129
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,250.0,1.0,0.0,0.8,7.216207,28.229730,28.276431,28.276431,28.274232
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,250.0,1.0,0.0,0.8,7.216207,26.238430,26.263232,26.263232,26.263967
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,250.0,1.0,0.0,0.8,7.216207,24.247130,24.260344,24.260344,24.258749
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,250.0,1.0,0.0,0.8,21.360697,6.408465,6.408659,6.408659,6.409880
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,250.0,1.0,0.0,0.8,21.360697,6.054016,6.054170,6.054170,6.054926
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,250.0,1.0,0.0,0.8,21.360697,5.857477,5.857625,5.857625,5.858013
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,250.0,1.0,0.0,0.8,21.360697,5.797751,5.797938,5.797938,5.798044


In [9]:
# Input CONV TEMP 04

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_temp_conv04.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_conv04':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,1.0,0.0,0.8,7.216207,29.567320,29.351169,29.351169,29.351169,29.351169
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,1.0,0.0,0.8,7.216207,28.898525,28.984686,28.984686,28.984129,28.984129
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,1.0,0.0,0.8,7.216207,28.229730,28.276431,28.276431,28.274232,28.274232
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,1.0,0.0,0.8,7.216207,26.238430,26.263232,26.263232,26.263967,26.263967
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,250.0,1.0,0.0,0.8,7.216207,24.247130,24.260344,24.260344,24.258749,24.258749
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,250.0,1.0,0.0,0.8,21.360697,6.408465,6.408659,6.408659,6.409880,6.409880
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,250.0,1.0,0.0,0.8,21.360697,6.054016,6.054170,6.054170,6.054926,6.054926
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,250.0,1.0,0.0,0.8,21.360697,5.857477,5.857625,5.857625,5.858013,5.858013
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,250.0,1.0,0.0,0.8,21.360697,5.797751,5.797938,5.797938,5.798044,5.798044


In [10]:
# Input MIX TEMP 05

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_temp_mix05.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_mix05':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.0,0.0,0.8,7.216207,29.567320,29.351169,29.351169,29.351169,29.351169,29.343814
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.0,0.0,0.8,7.216207,28.898525,28.984686,28.984686,28.984129,28.984129,28.992415
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.0,0.0,0.8,7.216207,28.229730,28.276431,28.276431,28.274232,28.274232,28.274232
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.0,0.0,0.8,7.216207,26.238430,26.263232,26.263232,26.263967,26.263967,26.263967
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.0,0.0,0.8,7.216207,24.247130,24.260344,24.260344,24.258749,24.258749,24.258749
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,1.0,0.0,0.8,21.360697,6.408465,6.408659,6.408659,6.409880,6.409880,6.409880
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,1.0,0.0,0.8,21.360697,6.054016,6.054170,6.054170,6.054926,6.054926,6.054926
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,1.0,0.0,0.8,21.360697,5.857477,5.857625,5.857625,5.858013,5.858013,5.858013
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,1.0,0.0,0.8,21.360697,5.797751,5.797938,5.797938,5.798044,5.798044,5.798044


In [11]:
# Input BUOYANCY

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_buoyancy.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'buoyancy':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.8,7.216207,29.567320,29.351169,29.351169,29.351169,29.351169,29.343814,0.002034
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.8,7.216207,28.898525,28.984686,28.984686,28.984129,28.984129,28.992415,0.004089
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.8,7.216207,28.229730,28.276431,28.276431,28.274232,28.274232,28.274232,0.010957
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.8,7.216207,26.238430,26.263232,26.263232,26.263967,26.263967,26.263967,0.010193
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.0,0.8,7.216207,24.247130,24.260344,24.260344,24.258749,24.258749,24.258749,0.015120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.8,21.360697,6.408465,6.408659,6.408659,6.409880,6.409880,6.409880,0.000243
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.8,21.360697,6.054016,6.054170,6.054170,6.054926,6.054926,6.054926,0.000119
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.8,21.360697,5.857477,5.857625,5.857625,5.858013,5.858013,5.858013,0.000034
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.0,0.8,21.360697,5.797751,5.797938,5.797938,5.798044,5.798044,5.798044,0.000014


In [12]:
# Input DIFFUSIVITY

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_diff.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'diffusivity':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.8,7.216207,29.567320,29.351169,29.351169,29.351169,29.351169,29.343814,0.002034,1.418698e-07
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.8,7.216207,28.898525,28.984686,28.984686,28.984129,28.984129,28.992415,0.004089,1.404211e-07
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.8,7.216207,28.229730,28.276431,28.276431,28.274232,28.274232,28.274232,0.010957,1.400488e-07
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.8,7.216207,26.238430,26.263232,26.263232,26.263967,26.263967,26.263967,0.010193,1.400048e-07
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,0.8,7.216207,24.247130,24.260344,24.260344,24.258749,24.258749,24.258749,0.015120,1.400004e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.8,21.360697,6.408465,6.408659,6.408659,6.409880,6.409880,6.409880,0.000243,2.800001e-07
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.8,21.360697,6.054016,6.054170,6.054170,6.054926,6.054926,6.054926,0.000119,2.800000e-07
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.8,21.360697,5.857477,5.857625,5.857625,5.858013,5.858013,5.858013,0.000034,2.800000e-07
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.8,21.360697,5.797751,5.797938,5.797938,5.798044,5.798044,5.798044,0.000014,2.800000e-07


In [13]:
# Input density gradient

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_density-conv.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'densityGradient':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity,densityGradient
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,29.567320,29.351169,29.351169,29.351169,29.351169,29.343814,0.002034,1.418698e-07,-0.108094
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,28.898525,28.984686,28.984686,28.984129,28.984129,28.992415,0.004089,1.404211e-07,-0.205666
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,28.229730,28.276431,28.276431,28.274232,28.274232,28.274232,0.010957,1.400488e-07,-0.557639
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,26.238430,26.263232,26.263232,26.263967,26.263967,26.263967,0.010193,1.400048e-07,-0.518754
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,7.216207,24.247130,24.260344,24.260344,24.258749,24.258749,24.258749,0.015120,1.400004e-07,-0.769503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,21.360697,6.408465,6.408659,6.408659,6.409880,6.409880,6.409880,0.000243,2.800001e-07,-0.012350
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,21.360697,6.054016,6.054170,6.054170,6.054926,6.054926,6.054926,0.000119,2.800000e-07,-0.006036
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,21.360697,5.857477,5.857625,5.857625,5.858013,5.858013,5.858013,0.000034,2.800000e-07,-0.001722
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,21.360697,5.797751,5.797938,5.797938,5.798044,5.798044,5.798044,0.000014,2.800000e-07,0.000726


In [14]:
# Input temp change

out_temp_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_temp-conv.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'tempChange':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity,densityGradient,tempChange
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,29.567320,29.351169,29.351169,29.351169,29.351169,29.343814,0.002034,1.418698e-07,-0.108094,28.984129
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,28.898525,28.984686,28.984686,28.984129,28.984129,28.992415,0.004089,1.404211e-07,-0.205666,28.274232
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,28.229730,28.276431,28.276431,28.274232,28.274232,28.274232,0.010957,1.400488e-07,-0.557639,26.263967
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,26.238430,26.263232,26.263232,26.263967,26.263967,26.263967,0.010193,1.400048e-07,-0.518754,24.258749
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,24.247130,24.260344,24.260344,24.258749,24.258749,24.258749,0.015120,1.400004e-07,-0.769503,20.954006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,6.408465,6.408659,6.408659,6.409880,6.409880,6.409880,0.000243,2.800001e-07,-0.012350,6.054926
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,6.054016,6.054170,6.054170,6.054926,6.054926,6.054926,0.000119,2.800000e-07,-0.006036,5.858013
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,5.857477,5.857625,5.857625,5.858013,5.858013,5.858013,0.000034,2.800000e-07,-0.001722,5.798044
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,5.797751,5.797938,5.797938,5.798044,5.798044,5.798044,0.000014,2.800000e-07,0.000726,5.823573


In [15]:
# ICE AND SNOW

ice_data_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_icesnow.csv")

#repeating the dataframe depth_steps number of times
ice_data_df = pd.DataFrame(np.repeat(ice_data_df.values, depth_steps, axis=0), columns=ice_data_df.columns)
ice_data_df = pd.concat([depth_df, ice_data_df], ignore_index=False, axis=1)
print(ice_data_df)

final_df = final_df.merge(ice_data_df, how='inner', on=['time','depth'])
final_df

        depth                 time  ice snow snowice
0        0.25  2018-07-05 15:00:00  0.0  0.0     0.0
1        0.75  2018-07-05 15:00:00  0.0  0.0     0.0
2        1.25  2018-07-05 15:00:00  0.0  0.0     0.0
3        1.75  2018-07-05 15:00:00  0.0  0.0     0.0
4        2.25  2018-07-05 15:00:00  0.0  0.0     0.0
...       ...                  ...  ...  ...     ...
788395   6.75  2023-07-04 14:00:00  0.0  0.0     0.0
788396   7.25  2023-07-04 14:00:00  0.0  0.0     0.0
788397   7.75  2023-07-04 14:00:00  0.0  0.0     0.0
788398   8.25  2023-07-04 14:00:00  0.0  0.0     0.0
788399   8.75  2023-07-04 14:00:00  0.0  0.0     0.0

[788400 rows x 5 columns]


,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity,densityGradient,tempChange,ice,snow,snowice
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,29.351169,29.351169,29.343814,0.002034,1.418698e-07,-0.108094,28.984129,0.0,0.0,0.0
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,28.984129,28.984129,28.992415,0.004089,1.404211e-07,-0.205666,28.274232,0.0,0.0,0.0
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,28.274232,28.274232,28.274232,0.010957,1.400488e-07,-0.557639,26.263967,0.0,0.0,0.0
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,26.263967,26.263967,26.263967,0.010193,1.400048e-07,-0.518754,24.258749,0.0,0.0,0.0
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,24.258749,24.258749,24.258749,0.015120,1.400004e-07,-0.769503,20.954006,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,6.409880,6.409880,6.409880,0.000243,2.800001e-07,-0.012350,6.054926,0.0,0.0,0.0
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,6.054926,6.054926,6.054926,0.000119,2.800000e-07,-0.006036,5.858013,0.0,0.0,0.0
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,5.858013,5.858013,5.858013,0.000034,2.800000e-07,-0.001722,5.798044,0.0,0.0,0.0
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,5.798044,5.798044,5.798044,0.000014,2.800000e-07,0.000726,5.823573,0.0,0.0,0.0


In [16]:
# lake characteristics

ice_data_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_lakecharacteristics.csv")

#repeating the dataframe depth_steps number of times
ice_data_df = pd.DataFrame(np.repeat(ice_data_df.values, depth_steps, axis=0), columns=ice_data_df.columns)
ice_data_df = pd.concat([depth_df, ice_data_df], ignore_index=False, axis=1)
print(ice_data_df)

final_df = final_df.merge(ice_data_df, how='inner', on=['time','depth'])
final_df

        depth                 time      Volume_m2    Osgood MaxDepth_m  \
0        0.25  2018-07-05 15:00:00  322278.388607  8.764344       9.75   
1        0.75  2018-07-05 15:00:00  322278.388607  8.764344       9.75   
2        1.25  2018-07-05 15:00:00  322278.388607  8.764344       9.75   
3        1.75  2018-07-05 15:00:00  322278.388607  8.764344       9.75   
4        2.25  2018-07-05 15:00:00  322278.388607  8.764344       9.75   
...       ...                  ...            ...       ...        ...   
788395   6.75  2023-07-04 14:00:00  322278.388607  8.764344       9.75   
788396   7.25  2023-07-04 14:00:00  322278.388607  8.764344       9.75   
788397   7.75  2023-07-04 14:00:00  322278.388607  8.764344       9.75   
788398   8.25  2023-07-04 14:00:00  322278.388607  8.764344       9.75   
788399   8.75  2023-07-04 14:00:00  322278.388607  8.764344       9.75   

       MeanDepth_m  
0         3.688313  
1         3.688313  
2         3.688313  
3         3.688313  
4     

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,diffusivity,densityGradient,tempChange,ice,snow,snowice,Volume_m2,Osgood,MaxDepth_m,MeanDepth_m
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.418698e-07,-0.108094,28.984129,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.404211e-07,-0.205666,28.274232,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.400488e-07,-0.557639,26.263967,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.400048e-07,-0.518754,24.258749,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,1.400004e-07,-0.769503,20.954006,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,2.800001e-07,-0.012350,6.054926,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,2.800000e-07,-0.006036,5.858013,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,2.800000e-07,-0.001722,5.798044,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,2.800000e-07,0.000726,5.823573,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313


In [17]:
temp_obs_df = pd.read_csv("./../output/lakes/fallingcreek/output/py_observed_temp.csv")


flattened_temp = temp_obs_df.iloc[:,1:].to_numpy().flatten()
time_stamp = temp_obs_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'obs_temp':flattened_temp, 'depth':depth_list}

temp_obs_df = pd.DataFrame(data=data)

temp_obs_df


final_df = final_df.merge(temp_obs_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,densityGradient,tempChange,ice,snow,snowice,Volume_m2,Osgood,MaxDepth_m,MeanDepth_m,obs_temp
0,0.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,-0.108094,28.984129,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313,29.567320
1,0.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,-0.205666,28.274232,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313,28.898525
2,1.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,-0.557639,26.263967,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313,28.229730
3,1.75,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,-0.518754,24.258749,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313,26.238430
4,2.25,2018-07-05 15:00:00,21.750375,-22.828779,-134.955775,-40.234864,163.130618,1.3,327.997438,0.007725,...,-0.769503,20.954006,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313,24.247130
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
788395,6.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,-0.012350,6.054926,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313,10.872310
788396,7.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,-0.006036,5.858013,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313,10.876860
788397,7.75,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,-0.001722,5.798044,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313,10.867625
788398,8.25,2023-07-04 14:00:00,23.2892,-70.917406,-12.27592,1.072818,544.901536,1.3,4.862209,0.000466,...,0.000726,5.823573,0.0,0.0,0.0,322278.388607,8.764344,9.75,3.688313,10.858390


In [18]:
obs_array = final_df['obs_temp']
obs_array[obs_array == -999] = final_df['temp_mix05']
print(obs_array)
final_df['obs_temp'] = obs_array

0         29.567320
1         28.898525
2         28.229730
3         26.238430
4         24.247130
            ...    
788395    10.872310
788396    10.876860
788397    10.867625
788398    10.858390
788399    10.848445
Name: obs_temp, Length: 788400, dtype: float64


C:\Users\au740615\AppData\Local\Temp\ipykernel_37684\349148568.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obs_array[obs_array == -999] = final_df['temp_mix05']


In [19]:
final_df_null = final_df.fillna('')
print(final_df_null.head)

<bound method NDFrame.head of         depth                 time  AirTemp_degC  Longwave_Wm-2  Latent_Wm-2  \
0        0.25  2018-07-05 15:00:00     21.750375     -22.828779  -134.955775   
1        0.75  2018-07-05 15:00:00     21.750375     -22.828779  -134.955775   
2        1.25  2018-07-05 15:00:00     21.750375     -22.828779  -134.955775   
3        1.75  2018-07-05 15:00:00     21.750375     -22.828779  -134.955775   
4        2.25  2018-07-05 15:00:00     21.750375     -22.828779  -134.955775   
...       ...                  ...           ...            ...          ...   
788395   6.75  2023-07-04 14:00:00     23.289200     -70.917406   -12.275920   
788396   7.25  2023-07-04 14:00:00     23.289200     -70.917406   -12.275920   
788397   7.75  2023-07-04 14:00:00     23.289200     -70.917406   -12.275920   
788398   8.25  2023-07-04 14:00:00     23.289200     -70.917406   -12.275920   
788399   8.75  2023-07-04 14:00:00     23.289200     -70.917406   -12.275920   

        S

In [20]:
# iterating the columns
for col in final_df_null.columns:
    print(col)

depth
time
AirTemp_degC
Longwave_Wm-2
Latent_Wm-2
Sensible_Wm-2
Shortwave_Wm-2
lightExtinct_m-1
TKE_Jm-1
ShearStress_Nm-2
Area_m2
CC
ea
Jlw
Uw
Pa
RH
PP
IceSnowAttCoeff
iceFlag
icemovAvg
density_snow
ice_prior
snow_prior
snowice_prior
rho_snow_prior
IceSnowAttCoeff_prior
iceFlag_prior
dt_iceon_avg_prior
icemovAvg_prior
temp_init00
temp_heat01
temp_ice02
temp_diff03
temp_conv04
temp_mix05
buoyancy
diffusivity
densityGradient
tempChange
ice
snow
snowice
Volume_m2
Osgood
MaxDepth_m
MeanDepth_m
obs_temp


In [21]:
final_df_null.to_csv("fallingcreek-all_data_lake_modeling_in_time.csv", index=False)